In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import warnings

warnings.filterwarnings("ignore")

from embedding import BM25_Plus
from utils.constants import BATCH_SIZE, NEAREST_NEIGHBOUR_COUNT
from tqdm import tqdm
from pathlib import Path
from chunking import chunk_files
from embedding import FAISS
from evaluator import Evaluator, SearchQueryData
from utils.embed_input import datadict2embed_input, jsonl2datadict
from pprint import pprint
import json

curr_dir = Path.cwd().parent / "data" / "test-repo"
print(curr_dir)
with open("output.jsonl", "w") as f:
    chunk_files(curr_dir, f)

with open("output.jsonl", "r") as f:
    lines = f.readlines()
    n_chunks = len(lines)
    metadatas = list(map(jsonl2datadict, lines))

target: list[SearchQueryData] = []
with open("../rag_eval_dataset_symbolic.jsonl", "r") as f:
    for line in f.readlines():
        target.append(json.loads(line))

texts: list[str] = []
for i in range(n_chunks):
    embed_input = datadict2embed_input(metadatas[i])
    texts.append(embed_input)

/home/gaurav/coding/SoftwareEngineeringAgent/data/test-repo


In [34]:
dense = FAISS(batch_size=BATCH_SIZE)
dense.embed_and_store(texts, metadatas)

Embedding and saving batches: 


100%|██████████| 95/95 [00:03<00:00, 27.88it/s]


In [31]:
from embedding import BM25_Plus
bm25 = BM25_Plus(k=0.8, b=0.85)
bm25.fit_finalize(texts, list(range(len(texts))), metadatas)

In [32]:
bm25.inverted_idx["controller"]

{70: 2,
 71: 1,
 72: 2,
 73: 2,
 74: 2,
 75: 2,
 76: 2,
 77: 2,
 78: 2,
 79: 2,
 80: 2,
 81: 2,
 82: 4,
 96: 2,
 97: 2,
 99: 2,
 100: 2,
 158: 1,
 161: 1,
 162: 1,
 163: 1,
 165: 1,
 167: 3}

In [33]:
res = bm25.search("UserControllersClass", )

for i, r in enumerate(res):
    print(i, r.qualified_name, r.file_name, r.chunk_type, r.chunk_id)

0 module.UserControllersClass._start_connection user_controller.py method 4295f8e4-0e3b-4aed-a585-408a327e49c2
1 module.UserControllersClass._close_connection user_controller.py method b0170062-3609-42dd-b7c4-ced6e2072904
2 module.UserControllersClass._validate_login_input user_controller.py method 249f8fc8-4dc6-4fdb-bded-2bd3a121ea42
3 module user_route.py module c3291883-f3f9-4277-9a49-0149d24fedd4
4 module.UserControllersClass._get_collection user_controller.py method 8801e42e-84ae-4e4e-a93a-402c14adbe78
5 module.UserControllersClass._store_user_session user_controller.py method c96163a3-c60d-4bdb-a5f9-c5eb013fe0d5
6 module.UserControllersClass.__init__ user_controller.py method 57e7ec39-df4d-4070-b8f6-41af8158cebc
7 module.UserControllersClass._store_refresh_token user_controller.py method ad2274b4-c8e3-4776-b6a8-0d9d58a65746
8 module.UserControllersClass._send_verification_email user_controller.py method 39977bbc-83d2-4650-b830-8514ff112456
9 module.UserControllersClass._validate_

In [131]:
res = bm25.search("User controller class")

for i, r in enumerate(res):
    pprint((i, r.qualified_name, r.file_name, r.chunk_type, r.chunk_id))

82 2815 ChunkMetaData(chunk_id='7fa9dfa3-8e14-4e0c-bbe8-839138bb49fc', chunk_type='class', name='UserControllersClass', qualified_name='module.UserControllersClass', parent_name='module', parent_type='module', file_path='backend/src/controllers/user_controller.py', file_name='user_controller.py', module_path='backend.src.controllers.user_controller', language='Python', start_line=15, end_line=421, source_code='class UserControllersClass:\n    def __init__(self):\n        self.Config = Settings()\n        self.mongo_db_connection = MongoDBConnection(self.Config.MONGO_URI, self.Config.DB_NAME)\n        self.email_services = EmailServices()\n        self.password_manager = PasswordManager()\n        self.jwt_manager = JWTManager()\n        self.email_regex_parrern = r"^(?!.*\\.\\.)[a-zA-Z0-9!#$%&\'*+/=?^_`{|}~-]+(?:\\.[a-zA-Z0-9!#$%&\'*+/=?^_`{|}~-]+)*@(?:[a-zA-Z0-9-]+\\.)+[a-zA-Z]{2,}$"\n\n    def _start_connection(self):\n        """\n        Start the MongoDB connection.\n        """\n

In [77]:
res

[ChunkMetaData(chunk_id='96d3d40f-1c44-4243-8fe2-5ee4ce3a6955', chunk_type='module', name='module', qualified_name='module', parent_name=None, parent_type=None, file_path='backend/src/routes/user_route.py', file_name='user_route.py', module_path='backend.src.routes.user_route', language='Python', start_line=0, end_line=13, source_code='from fastapi import APIRouter, Depends, BackgroundTasks, HTTPException, Request\nfrom fastapi.security import HTTPBearer, HTTPAuthorizationCredentials\nfrom src.schemas.user_schema import LoginUser, SignUpUser\nfrom src.controllers.user_controller import UserControllersClass\nfrom src.dependencies.user_auth_dependency import token_required\nuser_router = APIRouter()\nsecurity = HTTPBearer()\nuser_controllers = UserControllersClass()\n# Public routes\n', parameters=[], return_type=None, decorators=[], is_async=False, base_classes=[], imports=[Import(module_name='fastapi.APIRouter', alias='APIRouter'), Import(module_name='fastapi.Depends', alias='Depends')

In [54]:
from embedding import Tokenizer
t = Tokenizer()

t.tokenize("UserControlClass")

{'usercontrolclass': 1, 'user': 1, 'control': 1, 'class': 1}